<a href="https://colab.research.google.com/github/njshrutics24-bot/AI/blob/main/1BF24CS185_Week9_FOL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# N J SHRUTI
#1BF24CS185

class Term:
    def __init__(self, name, is_var=False):
        self.name = name
        self.is_var = is_var

    def __repr__(self):
        return self.name

    def __eq__(self, other):
        return isinstance(other, Term) and self.name == other.name and self.is_var == other.is_var

    def __hash__(self):
        return hash((self.name, self.is_var))


class Predicate:
    def __init__(self, name, args, negated=False):
        self.name = name
        self.args = args
        self.negated = negated

    def __repr__(self):
        sign = "~" if self.negated else ""
        return f"{sign}{self.name}({', '.join(map(str, self.args))})"

    def __eq__(self, other):
        return (
            isinstance(other, Predicate)
            and self.name == other.name
            and self.args == other.args
            and self.negated == other.negated
        )

    def __hash__(self):
        return hash((self.name, tuple(self.args), self.negated))


class Not:
    def __init__(self, formula):
        self.formula = formula

    def __repr__(self):
        return f"~({self.formula})"


class And:
    def __init__(self, left, right):
        self.left = left
        self.right = right

    def __repr__(self):
        return f"({self.left} ^ {self.right})"


class Or:
    def __init__(self, left, right):
        self.left = left
        self.right = right

    def __repr__(self):
        return f"({self.left} V {self.right})"


class Implies:
    def __init__(self, left, right):
        self.left = left
        self.right = right

    def __repr__(self):
        return f"({self.left} -> {self.right})"


class ForAll:
    def __init__(self, var, formula):
        self.var = var
        self.formula = formula

    def __repr__(self):
        return f"ForAll {self.var} [{self.formula}]"


def eliminate_implications(f):
    if isinstance(f, Predicate):
        return f
    if isinstance(f, Not):
        return Not(eliminate_implications(f.formula))
    if isinstance(f, And):
        return And(eliminate_implications(f.left), eliminate_implications(f.right))
    if isinstance(f, Or):
        return Or(eliminate_implications(f.left), eliminate_implications(f.right))
    if isinstance(f, ForAll):
        return ForAll(f.var, eliminate_implications(f.formula))
    if isinstance(f, Implies):
        return Or(Not(eliminate_implications(f.left)), eliminate_implications(f.right))
    return f


def move_negation_inwards(f):
    if isinstance(f, Predicate):
        return f
    if isinstance(f, And):
        return And(move_negation_inwards(f.left), move_negation_inwards(f.right))
    if isinstance(f, Or):
        return Or(move_negation_inwards(f.left), move_negation_inwards(f.right))
    if isinstance(f, ForAll):
        return ForAll(f.var, move_negation_inwards(f.formula))
    if isinstance(f, Not):
        inner = f.formula
        if isinstance(inner, Not):
            return move_negation_inwards(inner.formula)
        if isinstance(inner, Predicate):
            return Predicate(inner.name, inner.args, not inner.negated)
        if isinstance(inner, And):
            return Or(move_negation_inwards(Not(inner.left)), move_negation_inwards(Not(inner.right)))
        if isinstance(inner, Or):
            return And(move_negation_inwards(Not(inner.left)), move_negation_inwards(Not(inner.right)))
    return f


var_counter = 1


def standardize_variables(f, mapping=None):
    global var_counter
    if mapping is None:
        mapping = {}

    if isinstance(f, ForAll):
        new_var = Term(f"{f.var.name}{var_counter}", is_var=True)
        var_counter += 1
        new_mapping = mapping.copy()
        new_mapping[f.var.name] = new_var
        return ForAll(new_var, standardize_variables(f.formula, new_mapping))
    if isinstance(f, Predicate):
        new_args = [mapping[arg.name] if arg.is_var and arg.name in mapping else arg for arg in f.args]
        return Predicate(f.name, new_args, f.negated)
    if isinstance(f, Not):
        return Not(standardize_variables(f.formula, mapping))
    if isinstance(f, And):
        return And(standardize_variables(f.left, mapping), standardize_variables(f.right, mapping))
    if isinstance(f, Or):
        return Or(standardize_variables(f.left, mapping), standardize_variables(f.right, mapping))
    return f


def drop_universal_quantifiers(f):
    if isinstance(f, ForAll):
        return drop_universal_quantifiers(f.formula)
    if isinstance(f, Predicate):
        return f
    if isinstance(f, Not):
        return Not(drop_universal_quantifiers(f.formula))
    if isinstance(f, And):
        return And(drop_universal_quantifiers(f.left), drop_universal_quantifiers(f.right))
    if isinstance(f, Or):
        return Or(drop_universal_quantifiers(f.left), drop_universal_quantifiers(f.right))
    return f


def extract_clauses(f):
    if isinstance(f, Predicate):
        return [frozenset([f])]
    if isinstance(f, And):
        return extract_clauses(f.left) + extract_clauses(f.right)
    if isinstance(f, Or):
        result = []
        for lc in extract_clauses(f.left):
            for rc in extract_clauses(f.right):
                result.append(lc.union(rc))
        return result
    return []


def occurs_check(var, x, theta):
    if var == x:
        return True
    if x in theta:
        return occurs_check(var, theta[x], theta)
    return False


def unify_var(var, x, theta):
    if var in theta:
        return unify(theta[var], x, theta)
    if x in theta:
        return unify(var, theta[x], theta)
    if occurs_check(var, x, theta):
        return None
    theta[var] = x
    return theta


def unify(x, y, theta=None):
    if theta is None:
        theta = {}
    if x == y:
        return theta
    if isinstance(x, Term) and x.is_var:
        return unify_var(x, y, theta)
    if isinstance(y, Term) and y.is_var:
        return unify_var(y, x, theta)
    if isinstance(x, list) and isinstance(y, list) and len(x) == len(y):
        for xi, yi in zip(x, y):
            theta = unify(xi, yi, theta)
            if theta is None:
                return None
        return theta
    return None


def substitute(clause, theta):
    new_lits = set()
    for lit in clause:
        new_args = []
        for arg in lit.args:
            while arg in theta:
                arg = theta[arg]
            new_args.append(arg)
        new_lits.add(Predicate(lit.name, new_args, lit.negated))
    return frozenset(new_lits)


def resolve(c1, c2):
    resolvents = []
    for l1 in c1:
        for l2 in c2:
            if l1.name == l2.name and l1.negated != l2.negated:
                theta = unify(l1.args, l2.args)
                if theta is not None:
                    rem1 = {l for l in c1 if l != l1}
                    rem2 = {l for l in c2 if l != l2}
                    res = substitute(rem1.union(rem2), theta)
                    resolvents.append((res, theta))
    return resolvents


def format_clause(c):
    return " V ".join(map(str, c)) if c else "NIL (Empty Clause)"


x = Term("x", is_var=True)
y = Term("y", is_var=True)
John = Term("John")
Peanuts = Term("Peanuts")
Apple = Term("Apple")
Vegetables = Term("Vegetables")
Anil = Term("Anil")
Harry = Term("Harry")

fol_kb = [
    ForAll(x, Implies(Predicate("Food", [x]), Predicate("Likes", [John, x]))),
    And(Predicate("Food", [Apple]), Predicate("Food", [Vegetables])),
    ForAll(x, ForAll(y, Implies(And(Predicate("Eats", [y, x]), Not(Predicate("Killed", [y]))), Predicate("Food", [x])))),
    And(Predicate("Eats", [Anil, Peanuts]), Predicate("Alive", [Anil])),
    ForAll(x, Implies(Predicate("Eats", [Anil, x]), Predicate("Eats", [Harry, x]))),
    ForAll(x, Implies(Predicate("Alive", [x]), Not(Predicate("Killed", [x])))),
    ForAll(x, Implies(Not(Predicate("Killed", [x])), Predicate("Alive", [x]))),
]

goal = Predicate("Likes", [John, Peanuts])

print("STEP 1: REPRESENTATION IN FOL")
for i, f in enumerate(fol_kb, 1):
    print(f"{i}. {f}")

step2_kb = [eliminate_implications(f) for f in fol_kb]
print("\nSTEP 2: ELIMINATE IMPLICATION")
for i, f in enumerate(step2_kb, 1):
    print(f"{i}. {f}")

step3_kb = [move_negation_inwards(f) for f in step2_kb]
print("\nSTEP 3: MOVE NEGATION INWARDS")
for i, f in enumerate(step3_kb, 1):
    print(f"{i}. {f}")

step4_kb = [standardize_variables(f) for f in step3_kb]
print("\nSTEP 4: STANDARDIZE VARIABLES")
for i, f in enumerate(step4_kb, 1):
    print(f"{i}. {f}")

step5_kb = [drop_universal_quantifiers(f) for f in step4_kb]
print("\nSTEP 5: DROP UNIVERSAL QUANTIFIERS")
for i, f in enumerate(step5_kb, 1):
    print(f"{i}. {f}")

clauses = []
for f in step5_kb:
    clauses.extend(extract_clauses(f))

negated_goal = frozenset([Predicate(goal.name, goal.args, negated=True)])
clauses.append(negated_goal)

print("\nSTEP 6: CLAUSE FORM (CNF) + NEGATED GOAL")
for idx, c in enumerate(clauses, 1):
    tag = " [Negated Goal]" if idx == len(clauses) else ""
    print(f"C{idx}: {format_clause(c)}{tag}")

print("\nSTEP 7: RESOLUTION PROOF")
seen = set(clauses)
proven = False
i = 0

while i < len(clauses) and not proven:
    for j in range(i):
        resolvents = resolve(clauses[i], clauses[j])
        for res, theta in resolvents:
            if len(res) == 0:
                print(f"Resolve C{i+1} & C{j+1} with {theta} ==> NIL (Empty Clause)")
                proven = True
                break
            if res not in seen:
                seen.add(res)
                clauses.append(res)
                print(f"C{len(clauses)}: {format_clause(res)} [from C{i+1} & C{j+1}, theta={theta}]")
        if proven:
            break
    i += 1

print("\nFINAL CONCLUSION:", "John likes peanuts is PROVEN!" if proven else "Not Proven.")


STEP 1: REPRESENTATION IN FOL
1. ForAll x [(Food(x) -> Likes(John, x))]
2. (Food(Apple) ^ Food(Vegetables))
3. ForAll x [ForAll y [((Eats(y, x) ^ ~(Killed(y))) -> Food(x))]]
4. (Eats(Anil, Peanuts) ^ Alive(Anil))
5. ForAll x [(Eats(Anil, x) -> Eats(Harry, x))]
6. ForAll x [(Alive(x) -> ~(Killed(x)))]
7. ForAll x [(~(Killed(x)) -> Alive(x))]

STEP 2: ELIMINATE IMPLICATION
1. ForAll x [(~(Food(x)) V Likes(John, x))]
2. (Food(Apple) ^ Food(Vegetables))
3. ForAll x [ForAll y [(~((Eats(y, x) ^ ~(Killed(y)))) V Food(x))]]
4. (Eats(Anil, Peanuts) ^ Alive(Anil))
5. ForAll x [(~(Eats(Anil, x)) V Eats(Harry, x))]
6. ForAll x [(~(Alive(x)) V ~(Killed(x)))]
7. ForAll x [(~(~(Killed(x))) V Alive(x))]

STEP 3: MOVE NEGATION INWARDS
1. ForAll x [(~Food(x) V Likes(John, x))]
2. (Food(Apple) ^ Food(Vegetables))
3. ForAll x [ForAll y [((~Eats(y, x) V Killed(y)) V Food(x))]]
4. (Eats(Anil, Peanuts) ^ Alive(Anil))
5. ForAll x [(~Eats(Anil, x) V Eats(Harry, x))]
6. ForAll x [(~Alive(x) V ~Killed(x))]
7. Fo